# Lecture 14 examples: Independence and sufficiency

STAT-S681 · Simulation-Based Inference

# Visualizing categorical random variables: Mosaic plot

Each column represents a category of $X$. Within each column, the boxes represent categories of $Y$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def mosaic(ax, tab, colors, row_names, title=""):
    """Columns are rows of tab (X); boxes within a column are columns of tab (Y)."""
    tab = np.asarray(tab, dtype=float)
    widths = tab.sum(axis=1) / tab.sum()
    left = 0.0
    centers = []
    for i in range(tab.shape[0]):
        if widths[i] > 0:
            heights = tab[i] / tab[i].sum()
            ax.bar(left, heights, width=widths[i], align="edge",
                   bottom=np.concatenate([[0], np.cumsum(heights)[:-1]]),
                   color=colors, edgecolor="white", linewidth=2)
        centers.append(left + widths[i] / 2)
        left += widths[i]
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks(centers, row_names)
    ax.set_title(title)

# P(X = 1) = 0.3, P(Y = 1) = 0.4, independent: P(X = x, Y = y) = P(X = x) P(Y = y)
p = np.outer([0.7, 0.3], [0.6, 0.4])
print(p)
fig, ax = plt.subplots(figsize=(5, 4.5))
mosaic(ax, p, ["lightgray", "steelblue"], ["X = 0", "X = 1"], "Independent X and Y")
plt.show()

# A mosaic showing dependence

Everyone aboard the Titanic, by sex ($X$) and survival ($Y$).

In [ ]:
# Rows: Male, Female. Columns: died, survived.
titanic = np.array([[1364, 367],
                    [126, 344]])
expected = np.outer(titanic.sum(axis=1), titanic.sum(axis=0)) / titanic.sum()
print(titanic)
print(expected.round())

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
mosaic(axes[0], expected, ["lightgray", "steelblue"], ["Male", "Female"],
       "If sex and survival were independent")
mosaic(axes[1], titanic, ["lightgray", "steelblue"], ["Male", "Female"], "Observed")
plt.show()

# A mosaic plot of simulated independent data

We can simulate independent samples to see how much variation to expect.

In [ ]:
rng = np.random.default_rng(1401)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, n in zip(axes, [200, 200, 2000]):
    x = rng.binomial(1, 0.3, n)
    y = rng.binomial(1, 0.4, n)
    tab = pd.crosstab(x, y).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    mosaic(ax, tab, ["lightgray", "steelblue"], ["X = 0", "X = 1"], f"n = {n}")
plt.show()

# Independence/dependence for continuous variables

Flipper length ($X$) and body mass ($Y$) for 342 penguins near Palmer Station, Antarctica. Shuffled: body masses randomly reassigned among the penguins.

In [ ]:
penguins = pd.read_csv("penguins.csv")
X = penguins["flipper_len"].to_numpy()
Y = penguins["body_mass"].to_numpy()
rng = np.random.default_rng(1403)
Y_shuffled = rng.permutation(Y)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, y, title in zip(axes, [Y_shuffled, Y], ["Shuffled", "As measured"]):
    ax.scatter(X, y, s=12, alpha=0.6, color="steelblue")
    ax.set_xlabel("Flipper length (mm)"); ax.set_ylabel("Body mass (g)")
    ax.set_title(title)
plt.show()

# Option 1: Discretize both variables and make a mosaic plot

Divide each axis into intervals $A_1,\ldots,A_4$ and $B_1,\ldots,B_4$ (here, at the sample quartiles).

In [ ]:
def bin4(v):
    return np.digitize(v, np.quantile(v, [0.25, 0.5, 0.75])) + 1   # bins 1..4

def table4(x, y):
    return pd.crosstab(bin4(x), bin4(y)).reindex(
        index=range(1, 5), columns=range(1, 5), fill_value=0)

tab_shuffled = table4(X, Y_shuffled)
tab_measured = table4(X, Y)

Each box should contain about $1/16$ of them. Let's check.

In [ ]:
print((tab_shuffled / tab_shuffled.values.sum()).round(3))
print((tab_measured / tab_measured.values.sum()).round(3))
print(1 / 16)

blues = ["#DCE7F2", "#A9C5E0", "#6A9BC8", "#2F6599"]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
mosaic(axes[0], tab_shuffled, blues, [1, 2, 3, 4], "Shuffled")
mosaic(axes[1], tab_measured, blues, [1, 2, 3, 4], "As measured")
plt.show()

# Option 2: Discretize only one variable and visualize the distribution of the other

Each panel approximates $Y \mid X \in A_i$.

In [ ]:
breaks = np.arange(2500, 6501, 250)
fig, axes = plt.subplots(2, 4, figsize=(12, 6), sharey=True)
for row, y in zip(axes, [Y_shuffled, Y]):
    for i, ax in enumerate(row, start=1):
        yi = y[bin4(X) == i]
        ax.hist(yi, bins=breaks, weights=np.ones(len(yi)) / len(yi), color="steelblue")
        ax.set_title(f"Flipper bin {i}"); ax.set_xlabel("Body mass (g)")
    row[0].set_ylabel("Proportion")
plt.tight_layout(); plt.show()

Top row: shuffled. Bottom row: as measured.

# The same information shown a different way

Each strip (a mirrored histogram) summarizes the distribution of $Y$ in one vertical strip of the scatterplot.

In [ ]:
def mirror_hist(ax, y, g, at, breaks, max_width, color="steelblue", edgecolor="white"):
    """Mirrored histograms of y within each group g, centred at positions `at`.
    Half-width is proportional to the proportion in each bin, on one scale."""
    groups = np.unique(g)
    props = [np.histogram(y[g == k], bins=breaks)[0] / np.sum(g == k) for k in groups]
    scale = max_width / max(p.max() for p in props)
    for k, p, x0 in zip(groups, props, at):
        w = p * scale
        nz = w > 0                                    # skip empty bins
        ax.bar(x0 - w[nz], np.diff(breaks)[nz], width=2 * w[nz], bottom=breaks[:-1][nz],
               align="edge", color=color, edgecolor=edgecolor)
        m = y[g == k].mean()
        ax.plot([x0 - max_width, x0 + max_width], [m, m], color="orange", lw=2)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, y in zip(axes, [Y_shuffled, Y]):
    mirror_hist(ax, y, bin4(X), at=[1, 2, 3, 4], breaks=breaks, max_width=0.45)
    ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("Flipper length bin")
    ax.set_ylabel("Body mass (g)")
plt.show()

# Conditioning on a categorical variable, penguin example

$Z$: species. Conditioning on $Z = \text{Gentoo}$ means throwing away every other row.

In [ ]:
species_col = {"Adelie": "steelblue", "Chinstrap": "darkorange", "Gentoo": "seagreen"}
gentoo = penguins[penguins["species"] == "Gentoo"]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharex=True, sharey=True)
for sp, col in species_col.items():
    rows = penguins["species"] == sp
    axes[0].scatter(X[rows], Y[rows], s=12, color=col, label=sp)
axes[0].legend(); axes[0].set_title("All penguins")
axes[1].scatter(gentoo["flipper_len"], gentoo["body_mass"], s=12, color="seagreen")
axes[1].set_title("Gentoo only")
for ax in axes:
    ax.set_xlabel("Flipper length (mm)"); ax.set_ylabel("Body mass (g)")
plt.show()

# The Gentoo rows, discretized

$X$: flipper length, cut into thirds within the species. $Y$: body mass, cut into thirds within the species.

In [ ]:
labels3 = ["Low", "Mid", "High"]
def bin3(v):
    return np.digitize(v, np.quantile(v, [1 / 3, 2 / 3]))   # 0, 1, 2

tab_gentoo = pd.crosstab(bin3(gentoo["flipper_len"]), bin3(gentoo["body_mass"]))
tab_gentoo.index = labels3; tab_gentoo.columns = labels3
print(tab_gentoo)

fig, ax = plt.subplots(figsize=(5, 4.5))
mosaic(ax, tab_gentoo, ["#C9DCEC", "#6FA0C8", "#2C5C85"], labels3, "Gentoo")
plt.show()

# Comparing conditional distributions

Each histogram represents $Y\mid X\in A_i,\ Z=\text{Gentoo}$.

In [ ]:
g = bin3(gentoo["flipper_len"])
mass = gentoo["body_mass"].to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), sharex=True)
for k, ax in enumerate(axes):
    ax.hist(mass[g == k], bins=np.arange(3875, 6376, 125), color="lightgray", edgecolor="black")
    ax.set_title(f"{labels3[k]} flipper length"); ax.set_xlabel("Body mass (g)")
plt.show()

# Example: Packing and posting a parcel

- $Z$: the parcel's weight.
- $X$: the time needed to pack it.
- $Y$: the postage charge.

At a fixed weight, the simulator generates the extra packing time and extra postage charges independently.

In [ ]:
rng = np.random.default_rng(1441)
N = 100_000
weight = rng.gamma(shape=2, scale=1, size=N)            # Z
packing = 2 + weight + rng.exponential(0.5, N)          # X
postage = 3 + 2 * weight + rng.exponential(1, N)        # Y

# Parcels in a weight range

We'll start with $z_0 = 2$ and $h = 1, .3, .05$.

In [ ]:
h_values = [1, 0.3, 0.05]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, h in zip(axes, h_values):
    keep = np.abs(weight - 2) < h
    ax.scatter(packing[keep], postage[keep], s=1, alpha=0.3, color="steelblue")
    ax.set_xlim(2.8, 9); ax.set_ylim(4.5, 16)
    ax.set_xlabel("Packing time (min)"); ax.set_ylabel("Postage ($)")
    ax.set_title(f"{2 - h:.2f} < weight < {2 + h:.2f} (n = {keep.sum()})")
plt.show()

# Postage within packing-time strips

Color by weight: the strips differ because they hold parcels of different weights.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, h in zip(axes, h_values):
    keep = np.abs(weight - 2) < h
    x, y, z = packing[keep], postage[keep], weight[keep]
    edges = np.linspace(np.quantile(x, 0.01), np.quantile(x, 0.99), 6)
    strip = np.digitize(x, edges[1:5])                   # five packing-time strips
    show = slice(0, 3000)
    ax.scatter(x[show], y[show], s=3, c=z[show], cmap="viridis",
               vmin=2 - h, vmax=2 + h)                   # color scale spans this range
    for e in edges[1:5]:
        ax.axvline(e, ls="--", color="black", lw=1)
    mirror_hist(ax, y, strip, at=(edges[:-1] + edges[1:]) / 2,
                breaks=np.arange(4, 30.01, 0.4), max_width=0.45 * (edges[1] - edges[0]),
                color="none", edgecolor="black")
    ax.set_xlim(2.8, 9); ax.set_ylim(4.5, 16)
    ax.set_xlabel("Packing time (min)"); ax.set_ylabel("Postage ($)")
    ax.set_title(f"{2 - h:.2f} < weight < {2 + h:.2f}")
plt.show()

# Example: Ten days of rain

- Day 1 is equally likely to be rainy or dry.
- Each subsequent day repeats the previous day's weather with probability $\theta$.
- Otherwise, the weather switches.

In [ ]:
def simulate_rain(theta, rng, days=10):
    theta = np.asarray(theta)
    n = len(theta)
    X = np.zeros((n, days), dtype=int)
    X[:, 0] = rng.binomial(1, 0.5, n)
    for t in range(1, days):
        repeat_day = rng.uniform(size=n) < theta
        X[:, t] = np.where(repeat_day, X[:, t - 1], 1 - X[:, t - 1])
    return X

# Simulated weather sequences

Each row is one simulation. Each column is one day.

In [ ]:
from matplotlib.colors import ListedColormap
rng = np.random.default_rng(1450)
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, theta in zip(axes, [0.2, 0.5, 0.9]):
    X_seq = simulate_rain(np.full(20, theta), rng)
    ax.imshow(X_seq, cmap=ListedColormap(["wheat", "steelblue"]), aspect="auto",
              extent=(0.5, 10.5, 20.5, 0.5))
    ax.set_xticks(range(1, 11)); ax.set_yticks([1, 5, 10, 15, 20])
    ax.set_xlabel("Day"); ax.set_ylabel("Simulation"); ax.set_title(f"theta = {theta}")
plt.show()

# $X_i$ and $\theta$ are independent

Suppose we plot $\theta$ against $X_1$, then $X_2$, and so on.

In [ ]:
rng = np.random.default_rng(1451)
N = 100_000
theta = rng.uniform(size=N)
X_rain = simulate_rain(theta, rng)

fig, axes = plt.subplots(2, 5, figsize=(12, 5), sharey=True)
for t, ax in enumerate(axes.flat):
    mirror_hist(ax, theta, X_rain[:, t], at=[0, 1], breaks=np.linspace(0, 1, 11),
                max_width=0.4)
    ax.set_xlim(-0.6, 1.6); ax.set_xticks([0, 1])
    ax.set_xlabel(f"X{t + 1}"); ax.set_title(f"Day {t + 1}")
for ax in axes[:, 0]:
    ax.set_ylabel("theta")
plt.tight_layout(); plt.show()

# But the entire vector $\mathbf X$ is not independent of $\theta$

Create a "total rainy days" statistic: $R(\mathbf X)=X_1+\cdots+X_{10}.$

In [ ]:
R = X_rain.sum(axis=1)
strip = np.digitize(theta, [0.2, 0.4, 0.6, 0.8])   # fifths of theta
show = slice(0, 3000)
jitter_R = R[show] + rng.uniform(-0.3, 0.3, 3000)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(theta[show], jitter_R, s=3, color="gray")
axes[1].scatter(theta[show], jitter_R, s=3, color="lightgray")
for v in [0.2, 0.4, 0.6, 0.8]:
    axes[1].axvline(v, ls="--", color="black", lw=1)
mirror_hist(axes[1], R, strip, at=np.arange(0.1, 1, 0.2),
            breaks=np.arange(-0.5, 10.6, 1), max_width=0.09)
for ax in axes:
    ax.set_xlabel("theta"); ax.set_ylabel("Total rainy days")
plt.show()

print(pd.Series(R).groupby(strip).agg(["mean", "std"]))

The distribution of the total changes with $\theta$.

# Random projections and numerical checks

1. Draw a direction $a$ and compute $T_a=a^\top\mathbf X$.
2. Compare the distributions of $T_a$ across values of $Y$.
3. Repeat for other directions.

For example, discretize $T_a$ and $Y$, then compare joint cell proportions with products of marginal proportions.

In [ ]:
def dependence_score(t, y):
    """Sum over cells of |joint proportion - product of marginal proportions|."""
    if len(np.unique(t)) <= 11:
        t_bin = t
    else:
        t_bin = np.digitize(t, np.unique(np.quantile(t, [0.25, 0.5, 0.75])))
    y_bin = np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))
    p = pd.crosstab(y_bin, t_bin).to_numpy() / len(t)
    return np.abs(p - np.outer(p.sum(axis=1), p.sum(axis=0))).sum()

coordinate_scores = [dependence_score(X_rain[:, t], theta) for t in range(10)]
rng = np.random.default_rng(1460)
A = rng.normal(size=(10, 50))                 # 50 random directions
projection_scores = [dependence_score(X_rain @ A[:, k], theta) for k in range(50)]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.scatter(coordinate_scores, np.zeros(10), color="steelblue")
ax.scatter(projection_scores, np.ones(50), color="steelblue")
ax.scatter([dependence_score(R, theta)], [2], color="steelblue")
ax.set_yticks([0, 1, 2], ["Each day X_t", "Random a'X", "Total R"])
ax.set_xlabel("Dependence score")
plt.tight_layout(); plt.show()